# Spark introduction
![image_1784972849817.png](/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/resources/image_1784972849817.png "image_1784972849817.png")

## Spark is Unified
**What does that mean?**

Spark is a unified platform for big data applications that supports many data processing tasks—such as data loading, SQL queries, machine learning, and streaming—using the same engine and consistent APIs. This unified approach makes it easier to combine different types of analytics in a single application while maintaining high performance through automatic optimization. For example, Spark can optimize SQL queries and machine learning tasks to process data more efficiently. Unlike earlier systems that required multiple tools and APIs, Spark provides one integrated platform, making it the standard for large-scale data processing. Over time, Spark has expanded its capabilities, particularly with its structured APIs (DataFrames, Datasets, and SQL), which improve optimization and simplify application development.

## Computing engine
Spark splits the compute with the Storage. You can have your data on Azure Storage or Amazon S3 and Spark can process it, wihout it storing anything. This allows you to have your dedicated storage that do not depend on Databricks or Spark. This is a big difference then for example Apache Hadoop, which contains the HDFS (Hadoop file system) and then the Mapreduce as it processing power. Spark can run on top of HDFS but does not depend only on it, while the mapreduce does.

## Libraries
While the core engine of Spark changes little since it's release, their libraries are constantly changing. We have the Spark SQL, MLlib, Spark Streaming and Structured Streaming and GraphX. We have many others.


# Why do we need Spark?
The need for a new data analytics engine and programming model, such as Apache Spark, arose because of major changes in computer hardware and data growth.

Before 2005, computers became faster mainly because processors increased in speed each year. Existing software automatically benefited from these improvements without requiring changes, so most applications were designed to run on a single processor.
Around 2005, processor speeds stopped increasing due to physical limits like heat dissipation. Instead, manufacturers added more CPU cores, making parallel computing necessary for better performance.
Meanwhile, data storage and data collection became much cheaper. Organizations could store massive amounts of data, while technologies like cameras, sensors, and sequencing machines produced increasingly large datasets at lower costs.
As a result, businesses began collecting far more data than traditional single-processor software could efficiently process.
This created the need for new programming models and analytics engines that could distribute work across many processors and machines. Apache Spark was designed to meet this need by enabling fast, large-scale parallel data processing.

# Spark Basic Architecture

Spark Standalone cluster manager, YARN or mesos - Utilizados para gerir os recursos que um cluster (Grupo de computadores) nos dá. Um grupo de computadores não nos da processamento paralelo sem uma framework por cima que divide o trabalho em tarefas e destina cada uma dessas tarefas para uma maquina. É isto que o YARN ou o mesos é responsável.

Spark, in addition to its cluster mode, also has a local mode. The driver and executors are simply processed, which means that they can live on the same machine or different machines. In local mode, the driver and executer run (as threads) on your individual computer instead of a cluster. 

The SparkSession object is the entrance point to run Spark Code. When using Spark from Python or R, you don't write explicit JVM instructions; instead, you write Python and R code that Sparks translates into code that it then can run on the executor JVMs.





In [0]:
myRange = spark.range(1000).toDF("number")
# The myRange variable is a distributed collection

## DataFrames
A DataFrame is the most common Structured API and simply represents a table of data with rows and columns. 

![image_1785495396083.png](/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/resources/image_1785495396083.png "image_1785495396083.png")

Spark has several core abstractions: Datasets, DataFrames, SQL Tables, and Resilient Distributed Datasets (RDD). These different abstractions all represent distributed collections of data.

### Partitions
To allow every executor to perform work in parallel, Spark breaks up the data into chuncks called partitions. A partition is a collection of rows that sit on one physical machine in your cluster. A DataFrame's partitions represent how the data is physically distributed acrosse the cluster of machines during execution. If you have one partition, Spark will have a parallelism of only one, even if you have thousands of executors. If you have many partitions but only one executor, Spark will still have a parallelism of only one because there is only one computation resource.

An important thing to note is that with DataFrames you do not (for the most part) manipulate partitions manually or individually. You simply specify high-level transformations of data in the physical partitions, and Spark determines how this work will actually execute on the cluster. Lower-level APIs do exist (via the RDD interface).

### Transformations
`divisBy2 = myRange.where("number % 2 = 0")`


There are 2 types of transformations: Those that specify narrow dependencies and those that specify wide dependencies.
Transformations consiting of narrow dependencies are those for which each input partition will contribute to only one output partition. In the preceding code snippet, the where statement specifies a narrow dependecy, where only one partitions contributes to at most one output partition.

![image_1785500129520.png](/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/resources/image_1785500129520.png "image_1785500129520.png")

A wide dependency (or wide transformation) style transformation will have input partitions contributing to many output partitions. You will often head this reffered to as **shuffle** whereby Spark will exchange partitions across the cluster. With narrow transformations, Spark will automatically perform an operation called **pipelining**, meaning that if we specify multiple filters on DataFrames, they'll all be performed in-memory. The same cannot be said for shuffles. When we perform a shuffle, Spark writes the result to disk.

![image_1785500300542.png](/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/resources/image_1785500300542.png "image_1785500300542.png")

### Lazy Evaluation
Lazy evaluation in Spark means that it delays executing operations until an action is required. Instead of immediately changing the data, Spark records a sequence of transformations and creates an execution plan. Before running the job, Spark optimizes this plan into an efficient physical execution strategy for the cluster. This allows optimizations such as predicate pushdown, where filters are applied as early as possible, reducing the amount of data that needs to be read and processed (for example, retrieving only the single required row instead of scanning the entire dataset).


You can check whether **predicate pushdown** is being used by inspecting Spark's execution plan with `df.explain(True)` or `EXPLAIN` in Spark SQL. If the physical plan contains a **`PushedFilters`** section with filter conditions, Spark has pushed the filters down to the data source. Predicate pushdown is supported by formats like **Parquet, ORC, Delta Lake, Iceberg, and many JDBC sources**, but not by **CSV, JSON, or text files**. Only simple filter conditions (such as `=`, `<`, `>`, `<=`, `>=`) are typically pushed down, while filters involving functions (e.g., `upper()` or `length()`) usually cannot be pushed down.


### Actions
**Transformations** in Spark define a logical plan for processing data but are **not executed immediately**. Computation begins only when an **action** is called. Actions trigger Spark to execute the transformations and produce a result. Common types of actions include:

* **Viewing data** (e.g., `show()`)
* **Collecting data** into native language objects (e.g., `collect()`)
* **Writing data** to external storage (e.g., `write()`)

For example, `count()` is an action that executes the transformation pipeline, performs the necessary computations (including narrow and wide transformations), and returns the result. The execution process can be viewed in the **Spark UI**.


### Schema inference
**Schema inference** is Spark's ability to automatically determine a dataset's schema, including **column names, data types, and nullability**, when reading data. It is convenient for exploring new datasets because you don't need to define the schema manually. However, it requires Spark to scan the data first, making it slower and potentially less accurate if the data contains inconsistencies. For production workloads, it is generally recommended to define the schema explicitly for better performance, reliability, and consistent data types.

In [0]:
flightData2015 = spark.read.option("inferSchema", "true").option("header", "true").csv("/Workspace/Users/jcrmendes97@gmail.com/Spark-The-Definitive-Guide/data/flight-data/csv/2015-summary.csv")

# This DataFrame have a set of columns with an unspecified number of rows. The reason the number of rows is unspecified is because reading data is a transformation, and is therefore a lazy operation. Spark peeked at only a couple of rows of data to try to guess what types each column should be.

flightData2015.take(3)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count=344)]

### Why is SORT a wide transformation and the filter is not?
Because, each partition of the dataframe can apply a .filter and return the result to the original dataframe, while, the sort forces that we take the entire dataframe (creating a shuffle operation) to order the result and then return the entire result.

A **shuffle** is the process of **redistributing data between partitions**, which often involves moving data across different nodes in a Spark cluster. It occurs during **wide transformations** such as `sort()`, `groupBy()`, `join()`, `distinct()`, and `repartition()`, where Spark must reorganize data so that related records are placed in the correct partitions. Because shuffling often requires network communication, disk I/O, and task coordination, it is one of the most expensive operations in Spark.


In [0]:
flightData2015.sort("count").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [count#11227 ASC NULLS FIRST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7305]
               +- PhotonShuffleExchangeSink rangepartitioning(count#11227 ASC NULLS FIRST, 16)
                  +- PhotonRowToColumnar
                     +- FileScan csv [DEST_COUNTRY_NAME#11225,ORIGIN_COUNTRY_NAME#11226,count#11227] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Workspace/Users/jcrmendes97@gmail.com/Spark-The-Definitive-Guide..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "5") #By default, when we perform a shuffle, Spark outputs 200 shuffle partitions. Let’s set this value to 5 to reduce the number of the output partitions from the shuffle
flightData2015.sort("count").take(2)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Singapore', count=1)]

This is what happens when we set the shuffle partitions to 5
![image_1785505271396.png](/Users/jcrmendes/Projects/jcrmendes_projects/T-Shaped-Engineer/spark/resources/image_1785505271396.png "image_1785505271396.png")

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "5")
display(flightData2015.sort("count"))

DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count
United States,Singapore,1
Saint Vincent and the Grenadines,United States,1
Burkina Faso,United States,1
United States,Estonia,1
United States,Namibia,1
Moldova,United States,1
United States,Georgia,1
Zambia,United States,1
United States,Croatia,1
Iraq,United States,1


## SQL vs Dataframes
The code below generates the extact same execution plan. So it's the same thing and it contains the same performance to run Spark.sql logic or Spark Dataframes.

In [0]:
sqlWay = spark.sql("""
    SELECT DESTCOUNTRYNAME, count(1)
    FROM flightdata2015
    GROUP BY DESTCOUNTRYNAME
""")
dataFrameWay = flightData2015.groupBy("DESTCOUNTRYNAME").count()
sqlWay.explain()
dataFrameWay.explain()